
# Taller 1B — Introducción práctica a Deep Learning con PyTorch

## Objetivo

Construir y entrenar una red neuronal multicapa con PyTorch, conectando los conceptos vistos en clase:

- Tensores y dimensiones.
- Transformaciones lineales.
- Funciones de activación.
- Redes neuronales multicapa.
- Logits, probabilidades y funciones de costo.
- Autograd y backpropagation.
- Optimización con descenso del gradiente.
- Mini-batches y `DataLoader`.
- Curvas de entrenamiento.
- Efecto del *learning rate*.
- Importancia de la no linealidad.

> **Modalidad:** notebook de completar.  
> Los bloques marcados con `TODO` deben ser completados por el estudiante.

---

## Problema

Trabajaremos inicialmente con un conjunto de datos binario no lineal.

La pregunta central será:

> **¿Por qué una red neuronal multicapa con funciones de activación puede resolver problemas que un modelo puramente lineal no puede resolver?**

Al final construiremos también un ejemplo multiclase para conectar los **logits** con `CrossEntropyLoss`.


## 0. Librerías y reproducibilidad

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)

print("PyTorch:", torch.__version__)



## 1. Del `numpy.ndarray` al tensor

Generaremos un conjunto de datos con dos características:

$$
X \in \mathbb{R}^{N \times 2}
$$

y una etiqueta binaria por observación.


In [ ]:

X_np, y_np = make_moons(
    n_samples=600,
    noise=0.18,
    random_state=42
)

plt.figure(figsize=(6, 5))
plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, alpha=0.75)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Dataset no lineal")
plt.show()



### Actividad 1

Convierta `X_np` y `y_np` a tensores de PyTorch.

Tenga en cuenta:

- Las entradas de una red normalmente se representan con `float32`.
- Para `CrossEntropyLoss`, las etiquetas deben ser enteros (`long`).


In [ ]:

# TODO 1.1: convertir X_np a tensor float32
X = ...

# TODO 1.2: convertir y_np a tensor long
y = ...

print("X:", type(X), X.dtype, X.shape)
print("y:", type(y), y.dtype, y.shape)



**Preguntas**

1. ¿Qué representa cada dimensión de `X.shape`?
2. ¿Por qué `X` y `y` utilizan tipos de datos diferentes?



## 2. Una transformación lineal

Una capa `nn.Linear` implementa:

$$
z = XW^T + b
$$

Construiremos una capa que recibe 2 características y produce 4 salidas.


In [ ]:

# TODO 2.1: crear una capa lineal con 2 entradas y 4 salidas
linear = ...

print(linear)
print("W shape:", linear.weight.shape)
print("b shape:", linear.bias.shape)



### Antes de ejecutar el siguiente bloque

Para un batch de 10 observaciones:

1. ¿Qué dimensión tiene la entrada?
2. ¿Qué dimensión espera que tenga la salida?
3. ¿Cuántos parámetros entrenables tiene esta capa?


In [ ]:

X_batch = X[:10]

# TODO 2.2: aplicar la capa lineal
z = ...

print("Entrada:", X_batch.shape)
print("Salida:", z.shape)



## 3. Funciones de activación

Una transformación lineal por sí sola no introduce no linealidad.

Estudiaremos tres activaciones comunes:

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

$$
\tanh(z)
$$

$$
\mathrm{ReLU}(z)=\max(0,z)
$$


In [ ]:

z_values = torch.linspace(-5, 5, 200)

# TODO 3.1: calcule las tres activaciones
sigmoid_values = ...
tanh_values = ...
relu_values = ...


In [ ]:

fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(z_values.numpy(), sigmoid_values.detach().numpy(), label="Sigmoid")
ax.plot(z_values.numpy(), tanh_values.detach().numpy(), label="Tanh")
ax.plot(z_values.numpy(), relu_values.detach().numpy(), label="ReLU")

ax.axhline(0, linewidth=0.8)
ax.axvline(0, linewidth=0.8)
ax.set_xlabel("z")
ax.set_ylabel("f(z)")
ax.set_title("Funciones de activación")
ax.legend()
ax.grid(alpha=0.25)
plt.show()



### Actividad 3

Responda:

1. ¿Qué rango de salida tiene `Sigmoid`?
2. ¿Qué ocurre con valores negativos en `ReLU`?
3. ¿Qué diferencia observa entre `Sigmoid` y `Tanh` alrededor de cero?
4. ¿Cuál de estas funciones introduce no linealidad?
5. ¿Por qué una red neuronal necesita funciones de activación entre capas lineales?



## 4. Construcción de una red multicapa

Construiremos una MLP con arquitectura:

```text
2 → 8 → ReLU → 8 → ReLU → 2
```

La última capa produce **2 logits**, uno por clase.

> No aplicaremos `Softmax` dentro del modelo porque posteriormente utilizaremos `CrossEntropyLoss`.


In [ ]:

class MLP(nn.Module):

    def __init__(self):
        super().__init__()

        # TODO 4.1
        self.layer1 = ...
        self.activation1 = ...

        # TODO 4.2
        self.layer2 = ...
        self.activation2 = ...

        # TODO 4.3
        self.output = ...

    def forward(self, x):

        # TODO 4.4: completar forward pass
        x = ...
        x = ...
        x = ...
        x = ...
        logits = ...

        return logits


In [ ]:

model = MLP()
print(model)



### Actividad 4

1. ¿Cuántas capas lineales tiene la red?
2. ¿Cuántas capas ocultas?
3. ¿Cuál es la dimensión de la entrada?
4. ¿Por qué la capa final tiene 2 salidas?
5. Calcule manualmente el número total de parámetros entrenables.


In [ ]:

# Verificación del número de parámetros
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Parámetros entrenables:", total_params)


## 5. Forward pass: de entradas a logits

In [ ]:

X_small = X[:5]

# TODO 5.1
logits = ...

print(logits)
print("Shape:", logits.shape)



Para cada observación obtenemos:

$$
[z_0, z_1]
$$

Estos valores son **logits**, no probabilidades.

### Actividad 5

1. Para la primera observación, ¿qué clase elegiría usando únicamente los logits?
2. Convierta los logits a probabilidades utilizando `torch.softmax`.


In [ ]:

# TODO 5.2: aplicar softmax sobre la dimensión de clases
probabilities = ...

print(probabilities)
print("Suma por fila:", probabilities.sum(dim=1))


In [ ]:

# TODO 5.3: obtener la clase predicha para cada observación
predictions = ...

print("Predicciones:", predictions)
print("Etiquetas:    ", y[:5])



## 6. Función de costo: Cross-Entropy

Para clasificación multiclase, PyTorch proporciona:

```python
nn.CrossEntropyLoss()
```

Esta función espera:

- **logits** como entrada;
- etiquetas enteras como objetivo.

Conceptualmente combina `LogSoftmax + NLLLoss`.


In [ ]:

criterion = nn.CrossEntropyLoss()

# TODO 6.1: calcular el loss usando directamente los logits
loss = ...

print("Loss:", loss.item())



### Preguntas

1. ¿Por qué no debemos pasar `probabilities` a `CrossEntropyLoss`?
2. ¿Qué debería ocurrir con el `loss` durante un entrenamiento exitoso?


## 7. Autograd y backpropagation

In [ ]:

# Reiniciamos un modelo para observar sus gradientes
model = MLP()
criterion = nn.CrossEntropyLoss()

X_small = X[:8]
y_small = y[:8]

logits = model(X_small)
loss = criterion(logits, y_small)

print("Loss antes de backward:", loss.item())



Antes de ejecutar `backward()`, inspeccione los gradientes.


In [ ]:

for name, parameter in model.named_parameters():
    print(name, "->", parameter.grad)



### Actividad 7

Ejecute backpropagation y vuelva a inspeccionar los gradientes.


In [ ]:

# TODO 7.1: calcular los gradientes
...

for name, parameter in model.named_parameters():
    print(name)
    print(parameter.grad)
    print()



Cada tensor almacenado en `.grad` representa una derivada como:

$$
\frac{\partial L}{\partial W}
$$

o

$$
\frac{\partial L}{\partial b}
$$

según el parámetro correspondiente.

**Preguntas**

1. ¿Qué método de PyTorch realizó backpropagation?
2. ¿Qué información contiene `.grad`?
3. ¿El gradiente tiene las mismas dimensiones que el parámetro al cual pertenece?


## 8. Actualización de parámetros

In [ ]:

model = MLP()
criterion = nn.CrossEntropyLoss()

# TODO 8.1: crear SGD con learning rate 0.1
optimizer = ...



Complete **una única iteración** del entrenamiento.


In [ ]:

logits = model(X_small)
loss = criterion(logits, y_small)

# TODO 8.2: reiniciar gradientes
...

# TODO 8.3: backpropagation
...

# TODO 8.4: actualizar parámetros
...



Ordene conceptualmente las operaciones anteriores:

```text
Forward → Loss → __________ → __________ → Update
```

¿Por qué debemos ejecutar `zero_grad()`?


## 9. Separación train/validation

In [ ]:

X_train_np, X_val_np, y_train_np, y_val_np = train_test_split(
    X_np,
    y_np,
    test_size=0.25,
    random_state=42,
    stratify=y_np
)

X_train = torch.tensor(X_train_np, dtype=torch.float32)
X_val = torch.tensor(X_val_np, dtype=torch.float32)

y_train = torch.tensor(y_train_np, dtype=torch.long)
y_val = torch.tensor(y_val_np, dtype=torch.long)

print(X_train.shape, X_val.shape)


## 10. Training loop — Full Batch


Complete el ciclo de entrenamiento utilizando **todos los datos de entrenamiento en cada actualización**.


In [ ]:

model_full = MLP()
criterion = nn.CrossEntropyLoss()

# TODO 10.1
optimizer = ...

num_epochs = 200
train_losses = []
val_losses = []

for epoch in range(num_epochs):

    # ===== TRAIN =====
    model_full.train()

    # TODO 10.2: forward
    train_logits = ...

    # TODO 10.3: loss
    train_loss = ...

    # TODO 10.4: reset gradients
    ...

    # TODO 10.5: backward
    ...

    # TODO 10.6: update
    ...

    # ===== VALIDATION =====
    model_full.eval()

    with torch.no_grad():
        # TODO 10.7
        val_logits = ...
        val_loss = ...

    train_losses.append(train_loss.item())
    val_losses.append(val_loss.item())


In [ ]:

plt.figure(figsize=(7, 5))
plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("Curvas de entrenamiento")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


In [ ]:

with torch.no_grad():
    val_logits = model_full(X_val)
    val_predictions = val_logits.argmax(dim=1)
    val_accuracy = (val_predictions == y_val).float().mean()

print("Validation accuracy:", val_accuracy.item())


## 11. Mini-batches con `TensorDataset` y `DataLoader`

In [ ]:

train_dataset = TensorDataset(X_train, y_train)

# TODO 11.1: crear DataLoader de batch_size=32 y shuffle=True
train_loader = ...

print("Número de mini-batches:", len(train_loader))



### Antes de continuar

Si tenemos aproximadamente 450 observaciones de entrenamiento y `batch_size=32`:

1. ¿Cuántas actualizaciones de parámetros espera por época?
2. ¿Por qué no es exactamente `450 / 32` en todos los batches?


In [ ]:

model_mb = MLP()
criterion = nn.CrossEntropyLoss()

# TODO 11.2
optimizer = ...

num_epochs = 200
epoch_losses = []

for epoch in range(num_epochs):

    model_mb.train()
    batch_losses = []

    for X_batch, y_batch in train_loader:

        # TODO 11.3: forward
        logits = ...

        # TODO 11.4: loss
        loss = ...

        # TODO 11.5: reset gradients
        ...

        # TODO 11.6: backward
        ...

        # TODO 11.7: update
        ...

        batch_losses.append(loss.item())

    epoch_losses.append(np.mean(batch_losses))


In [ ]:

plt.figure(figsize=(7, 5))
plt.plot(epoch_losses)
plt.xlabel("Época")
plt.ylabel("Loss promedio")
plt.title("Entrenamiento con mini-batches")
plt.grid(alpha=0.25)
plt.show()


## 12. Visualización de la frontera de decisión

In [ ]:

def plot_decision_boundary(model, X, y, title):
    x1_min, x1_max = X[:, 0].min().item() - 0.5, X[:, 0].max().item() + 0.5
    x2_min, x2_max = X[:, 1].min().item() - 0.5, X[:, 1].max().item() + 0.5

    xx1, xx2 = np.meshgrid(
        np.linspace(x1_min, x1_max, 300),
        np.linspace(x2_min, x2_max, 300)
    )

    grid = torch.tensor(
        np.c_[xx1.ravel(), xx2.ravel()],
        dtype=torch.float32
    )

    model.eval()
    with torch.no_grad():
        logits = model(grid)
        preds = logits.argmax(dim=1).numpy()

    zz = preds.reshape(xx1.shape)

    plt.figure(figsize=(7, 5))
    plt.contourf(xx1, xx2, zz, alpha=0.25)
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolor="k", alpha=0.75)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(title)
    plt.show()


In [ ]:

plot_decision_boundary(
    model_mb,
    X,
    y,
    "Frontera de decisión — MLP"
)



## 13. Experimento clave: ¿qué pasa si quitamos las activaciones?

Construya ahora una red con varias capas lineales, **pero sin ReLU**:

```text
2 → Linear(8) → Linear(8) → Linear(2)
```

Recuerde que:

$$
W_2(W_1x+b_1)+b_2
$$

continúa siendo una transformación lineal respecto a $x$.


In [ ]:

class LinearStack(nn.Module):

    def __init__(self):
        super().__init__()

        # TODO 13.1
        self.layer1 = ...
        self.layer2 = ...
        self.output = ...

    def forward(self, x):

        # TODO 13.2
        x = ...
        x = ...
        logits = ...

        return logits


In [ ]:

linear_model = LinearStack()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(linear_model.parameters(), lr=0.1)

for epoch in range(200):

    logits = linear_model(X_train)
    loss = criterion(logits, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

plot_decision_boundary(
    linear_model,
    X,
    y,
    "Frontera de decisión — varias capas lineales sin activación"
)



### Análisis

Compare las dos fronteras de decisión:

1. ¿Qué forma tiene la frontera aprendida por `LinearStack`?
2. ¿Qué forma puede aprender la MLP con `ReLU`?
3. Si `LinearStack` tiene varias capas, ¿por qué sigue comportándose como un modelo lineal?
4. ¿Cuál es entonces el papel fundamental de las funciones de activación?


## 14. Experimento con Learning Rate


Entrene la misma arquitectura utilizando:

```text
η = 0.0001
η = 0.1
η = 5.0
```

y compare las curvas de `loss`.


In [ ]:

def train_with_lr(lr, epochs=100):

    model = MLP()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    losses = []

    for epoch in range(epochs):

        logits = model(X_train)
        loss = criterion(logits, y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    return model, losses


In [ ]:

# TODO 14.1: entrenar los tres modelos
model_lr_small, loss_small = ...
model_lr_ok, loss_ok = ...
model_lr_large, loss_large = ...


In [ ]:

plt.figure(figsize=(8, 5))
plt.plot(loss_small, label="lr = 0.0001")
plt.plot(loss_ok, label="lr = 0.1")
plt.plot(loss_large, label="lr = 5.0")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("Efecto del learning rate")
plt.legend()
plt.grid(alpha=0.25)
plt.show()



### Análisis

1. ¿Cuál aprende más lentamente?
2. ¿Cuál presenta el comportamiento más estable?
3. ¿Algún entrenamiento oscila o diverge?
4. Relacione estas curvas con la actualización:

$$
	heta_{t+1}
=
	heta_t
-
\eta 
abla_	heta L
$$


## 15. Logits y Cross-Entropy en un problema multiclase


Ahora trabajaremos directamente con logits de un problema de **3 clases**.

Suponga que una red produjo:


In [ ]:

logits = torch.tensor([
    [2.0, 1.0, 0.1],
    [0.2, 0.3, 2.5],
    [1.2, 2.1, 0.4]
])

labels = torch.tensor([0, 0, 1])



### Actividad 15

1. Obtenga las probabilidades con Softmax.
2. Obtenga la clase predicha.
3. Calcule `CrossEntropyLoss`.
4. Identifique cuál observación debería contribuir más al costo y explique por qué.


In [ ]:

# TODO 15.1
probs = ...

# TODO 15.2
preds = ...

# TODO 15.3
criterion = ...
loss = ...

print("Probabilidades:")
print(probs)

print("\nPredicciones:", preds)
print("Etiquetas:    ", labels)
print("Loss:", loss.item())



## 16. Cierre

Complete con sus propias palabras:

1. Una capa `Linear` realiza _______________________________________.
2. Una función de activación permite _______________________________.
3. Los logits son _________________________________________________.
4. `CrossEntropyLoss` compara _____________________________________.
5. `backward()` calcula ____________________________________________.
6. El optimizador utiliza los gradientes para ______________________.
7. `zero_grad()` es necesario porque _______________________________.
8. Un mini-batch es ________________________________________________.
9. Sin funciones de activación, varias capas lineales ______________.
10. El *learning rate* controla ____________________________________.

---

### Flujo completo

Al terminar el taller debe poder interpretar el siguiente ciclo:

```text
X
↓
Linear
↓
Activation
↓
Linear
↓
Activation
↓
Linear
↓
Logits
↓
Loss
↓
Backward
↓
Gradientes
↓
Optimizer.step()
```
